In [1]:
"""
Created on Mon Sep 23 17:03:01 2021
@author: Oumbeg
"""

from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

from time import sleep
import os
import re
import requests
import pdfplumber
import sys

regulatorName = 'AG FSRCAG'
print("Running AG FSRCAG Web Scraping Tool v.1.0")

os.environ["PATH"] += r"C:\Program Files (x86)\gs\gs9.54.0\bin"
os.environ["PATH"] += r"C:\Program Files (x86)\gs\gs9.54.0\lib"

#Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'AG FSRCAG SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])

#Assigning the folders that are going to be used in the process
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder,'tempfolder')
os.chdir(scriptfolder)

#Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
	for temp_file in os.listdir(tempfolder):
		os.remove(os.path.join(tempfolder, temp_file))
else:
	os.mkdir(tempfolder)

regdict={'AG FSRCAG 1': 'https://www.fsrc.gov.ag/index.php/directories'}


chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
        "plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}


# Define functions

def findTel(mydata):
    pattern = re.compile('[(+0-9)]+')
    matcher = pattern.findall(mydata)
    listToStr = ' '.join([str(elem) for elem in matcher])
    return listToStr


def findUrl(mydata):
    regex = re.compile(r"(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'\".,<>?«»“”‘’]))")
    url = regex.findall(mydata)
    return ' '.join([str(elem) for elem in [x[0] for x in url]])

def findEmail(myData):
    """
    This function finds email in a string.
    :param myData: string
    :return: String
    """
    regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')
    email = regex.findall(myData)
    return ' '.join([str(elem) for elem in email])
	
# try to create an empty folder "tempfolder"
try:
    os.mkdir(tempfolder)
except:
    prevfiles=os.listdir(tempfolder)
    os.chdir(tempfolder)
    for prf in prevfiles:
        os.remove(prf)
    print('The directory tempfolder already exists.')
os.chdir(tempfolder)##only if files are going to be downloaded here

processdate=now.strftime('%Y-%m-%d')

patternTel = re.compile('(el.:|el\. |Tel:|Cell:|Tel.)')
patternFax = re.compile('(Fax.:|Fax:|Fax.|Fax)')

for reg in regdict:
    
    print('Working with {}'.format(reg))
    driver.get(regdict[reg])
    sleep(3)
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    bloc = soup.find("div",{"class":"left"})
    url = bloc.find_all('a', href=True)
    
    for i in range(1, len(url)+1):
        # download the PDFs from search using xpath
        driver.find_element(By.XPATH, '//*[@id="content"]/div[2]/div/div/div/div[1]/a[%d]'%i).click()
        
        while len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele])==0:
            print('Waiting for file to download')
            sleep(3)
            
        # Credit Unions    
        if i == 1:
            tables = []

            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)     
            with pdfplumber.open(filePath) as pdf:
                for page in pdf.pages:
                    table = page.extract_table()
                    tables.append(table)

            for k in range(len(tables)):
                df_Table = tables[k]
                
                for j in range(len(df_Table)):
                    if 'NAME OF INSTITUTION' not in df_Table[0][j].upper():
                        sqldict['Name'].append(' '.join([item.strip() for item in df_Table[0][j].splitlines() if item !='']))
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('7')
                        sqldict['RegCode'].append('FSRCAG')
                        sqldict['RegCtry'].append('AG')
                        sqldict['RegulationType'].append('Regulated')

                        web = ''
                        mail = ''
                        tel = ''
                        fax = ''
                        address = ''
                        
                        for el in df_Table[1][j].splitlines():
                            if findUrl(el) != '':
                                web = findUrl(el)
                            elif findEmail(el) != '':
                                mail = findEmail(el)
                            elif len(patternTel.findall(el)) != 0 and len(patternFax.findall(el)) != 0:
                                tel_Fax = el.split(patternFax.findall(el)[0])
                                fax = tel_Fax[1].strip()
                                tel = tel_Fax[0].replace(patternTel.findall(el)[0],'').replace(';','').strip()
                            elif len(patternTel.findall(el)) != 0:
                                tel = el.replace(patternTel.findall(el)[0],'').replace(';','').strip()
                            elif len(patternFax.findall(el)) != 0:
                                fax = el.replace(patternFax.findall(el)[0],'').strip()
                            else:
                                if len(el.strip())>1 and len(findTel(el))<2:
                                    address = address +' '+el.strip()
                                    
                        sqldict['Address_1'].append(address)
                        sqldict['Phone'].append(tel)
                        sqldict['Fax'].append(fax)
                        sqldict['Email'].append(mail)
                        
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
                                
            os.remove(filePath)
        
        # CMT Service Providers
        elif i == 2:
            
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       
            
            for k in range(tables.n):
                df_Table = tables[k].df
                
                for j in range(len(df_Table)):
                    if 'NAME OF INSTITUTION' not in df_Table[1][j].upper() and df_Table[1][j].strip() != '':
                        name = ' '.join([item.strip() for item in df_Table[1][j].splitlines() if item !=''])
                        sqldict['Name'].append(name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('1')
                        sqldict['RegCode'].append('FSRCAG')
                        sqldict['RegCtry'].append('AG')
                        sqldict['RegulationType'].append('Supervised')

                        web = ''
                        mail = ''
                        tel = ''
                        fax = ''
                        address = ''
                        
                        for el in df_Table[3][j].splitlines():
                            if 'Email: ' in el and 'or' in el:
                                mail = el.split('or')[0].replace('Email:','').strip()
                                web = el.split('or')[1].strip()
                            elif findUrl(el) != '':
                                web = findUrl(el)
                            elif findEmail(el) != '':
                                mail = findEmail(el).split()[0].replace(';','')
                            elif len(patternTel.findall(el)) != 0 and len(patternFax.findall(el)) != 0:
                                tel_Fax = el.split(patternFax.findall(el)[0])
                                fax = tel_Fax[1].strip()
                                tel = tel_Fax[0].replace(patternTel.findall(el)[0],'').replace(';','').strip().replace('Mobile:','/')
                            elif len(patternTel.findall(el)) != 0:
                                tel = el.replace(patternTel.findall(el)[0],'').replace(';','').strip().replace('Mobile:','/')
                            elif len(patternFax.findall(el)) != 0:
                                fax = el.replace(patternFax.findall(el)[0],'').strip()
                            else:
                                if len(el.strip())>1:
                                    address = address +' '+el.strip()
                                    
                        
                        sqldict['Address_1'].append(address)
                        sqldict['Phone'].append(tel)
                        sqldict['Fax'].append(fax)
                        sqldict['Email'].append(mail)
                        
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
                
            os.remove(filePath)
            
        # Banks   
        elif i == 3:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       
            
            for k in range(tables.n):
                df_Table = tables[k].df
                
                for j in range(len(df_Table)):
                    if 'NAME OF INSTITUTION' not in df_Table[0][j].upper() and df_Table[0][j].strip() != '':
                        name = ' '.join([item.strip() for item in df_Table[0][j].splitlines() if item !=''])
                        sqldict['Name'].append(name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('2')
                        sqldict['RegCode'].append('FSRCAG')
                        sqldict['RegCtry'].append('AG')
                        sqldict['RegulationType'].append('Supervised')

                        web = ''
                        mail = ''
                        tel = ''
                        fax = ''
                        address = ''
                        
                        for el in df_Table[2][j].splitlines():
                            if findUrl(el) != '':
                                web = findUrl(el)
                            elif findEmail(el) != '':
                                mail = findEmail(el)
                            elif len(patternTel.findall(el)) != 0 and len(patternFax.findall(el)) != 0:
                                tel_Fax = el.split(patternFax.findall(el)[0])
                                fax = tel_Fax[1].replace('No.','').strip()
                                tel = tel_Fax[0].replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternTel.findall(el)) != 0:
                                tel = el.replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternFax.findall(el)) != 0:
                                fax = el.replace(patternFax.findall(el)[0],'').replace('No.','').strip()
                            else:
                                if len(el.strip())>1 and len(findTel(el))<2:
                                    address = address +' '+el.strip()
                                    
                        
                        sqldict['Address_1'].append(address)
                        sqldict['Phone'].append(tel)
                        sqldict['Fax'].append(fax)
                        sqldict['Email'].append(mail)
                        
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
                                
            os.remove(filePath)
            
        # Insurance Agents    
        elif i == 4:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       
            
            for k in range(tables.n):
                df_Table = tables[k].df
                
                for j in range(len(df_Table)):
                    if 'NAME OF INSTITUTION' not in df_Table[0][j].upper() and df_Table[0][j].strip() != '' and df_Table[1][j].strip() != '' and 'DETAILS' not in df_Table[1][j].strip():
                        name = ' '.join([item.strip() for item in df_Table[0][j].split() if item !='' and item != 'Status:' and item != '(Registered)'])
                        sqldict['Name'].append(name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('3')
                        sqldict['RegCode'].append('FSRCAG')
                        sqldict['RegCtry'].append('AG')
                        sqldict['RegulationType'].append('Supervised')

                        web = ''
                        mail = ''
                        tel = ''
                        fax = ''
                        address = ''
                        
                        for el in df_Table[1][j].splitlines():
                            if 'Email: ' in el and 'Website:' in el:
                                mail = el.split('Website:')[0].replace('Email:','').strip()
                                web = el.split('Website:')[1].strip()
                            elif findUrl(el) != '':
                                web = findUrl(el)
                            elif findEmail(el) != '':
                                mail = findEmail(el)
                            elif len(patternTel.findall(el)) != 0 and len(patternFax.findall(el)) != 0:
                                tel_Fax = el.split(patternFax.findall(el)[0])
                                fax = tel_Fax[1].replace('No.','').strip()
                                tel = tel_Fax[0].replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternTel.findall(el)) != 0:
                                tel = el.replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternFax.findall(el)) != 0:
                                fax = el.replace(patternFax.findall(el)[0],'').replace('No.','').strip()
                            else:
                                if len(el.strip())>1:# and len(findTel(el))<2:
                                    address = address +' '+el.strip()
                                    
                        sqldict['Address_1'].append(address)
                        sqldict['Phone'].append(tel)
                        sqldict['Fax'].append(fax)
                        sqldict['Email'].append(mail)
                        
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
                        
            os.remove(filePath)
            
        # Insurance Brokers    
        elif i == 5:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       
            
            for k in range(tables.n):
                df_Table = tables[k].df
                
                for j in range(len(df_Table)):
                    if 'NAME OF INSTITUTION' not in df_Table[0][j].upper() and df_Table[0][j].strip() != '' and df_Table[1][j].strip() != '' and 'DETAILS' not in df_Table[1][j].strip():
                        name = ' '.join([item.strip() for item in df_Table[0][j].split() if item !='' and item != 'Status:' and item != '(Registered)'])
                        sqldict['Name'].append(name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('4')
                        sqldict['RegCode'].append('FSRCAG')
                        sqldict['RegCtry'].append('AG')
                        sqldict['RegulationType'].append('Supervised')
                        
                        web = ''
                        mail = ''
                        tel = ''
                        fax = ''
                        address = ''
                        
                        for el in df_Table[1][j].splitlines():
                            if 'Email: ' in el and 'Website:' in el:
                                mail = el.split('Website:')[0].replace('Email:','').strip()
                                web = el.split('Website:')[1].strip()
                            elif findUrl(el) != '':
                                web = findUrl(el)
                            elif findEmail(el) != '':
                                mail = findEmail(el)
                            elif len(patternTel.findall(el)) != 0 and len(patternFax.findall(el)) != 0:
                                tel_Fax = el.split(patternFax.findall(el)[0])
                                fax = tel_Fax[1].replace('No.','').strip()
                                tel = tel_Fax[0].replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternTel.findall(el)) != 0:
                                tel = el.replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternFax.findall(el)) != 0:
                                fax = el.replace(patternFax.findall(el)[0],'').replace('No.','').strip()
                            else:
                                if len(el.strip())>1:# and len(findTel(el))<2:
                                    address = address +' '+el.strip()
                                    
                        sqldict['Address_1'].append(address)
                        sqldict['Phone'].append(tel)
                        sqldict['Fax'].append(fax)
                        sqldict['Email'].append(mail)
                        
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
                                
            os.remove(filePath)
            
        # Insurance Companies    
        elif i == 6:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       
            
            for k in range(tables.n):
                df_Table = tables[k].df
                
                for j in range(len(df_Table)):
                    print(df_Table.head())
                    if 'NAME OF COMPANY' not in df_Table[0][j].upper() and df_Table[0][j].strip() != '' and df_Table[1][j].strip() != '' and 'DETAILS' not in df_Table[2][j].strip():
                        name = ' '.join([item.strip() for item in df_Table[0][j].split() if item !='' and item != 'Status:' and item != '(Active)' and item != 'tatus:' and item != 'Under Liquidation' and item != 'Under Judicial Management'])
                        if name.split()[0] == 'S':
                            name = name[2:]
                        sqldict['Name'].append(name)
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('5')
                        sqldict['RegCode'].append('FSRCAG')
                        sqldict['RegCtry'].append('AG')
                        sqldict['RegulationType'].append('Supervised')
                        
                        web = ''
                        mail = ''
                        tel = ''
                        fax = ''
                        address = ''
                        
                        for el in df_Table[2][j].splitlines():
                            if 'Email:' in el and 'Website:' in el:
                                mail = el.split('Website:')[0].replace('Email:','').strip()
                                #if len(el.split('or')[1].strip()) != 0:
                                web = el.split('Website:')[1].strip()
                            elif findUrl(el) != '':
                                web = findUrl(el)
                            elif findEmail(el) != '':
                                mail = findEmail(el)
                            elif len(patternTel.findall(el)) != 0 and len(patternFax.findall(el)) != 0:
                                tel_Fax = el.split(patternFax.findall(el)[0])
                                fax = tel_Fax[1].replace('No.','').strip()
                                tel = tel_Fax[0].replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternTel.findall(el)) != 0:
                                tel = el.replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternFax.findall(el)) != 0:
                                fax = el.replace(patternFax.findall(el)[0],'').replace('No.','').strip()
                            else:
                                if len(el.strip())>1 and 'AGENT:' not in el.upper():
                                    address = address +' '+el.strip()
                                    
                        sqldict['Address_1'].append(address)
                        sqldict['Phone'].append(tel)
                        sqldict['Fax'].append(fax)
                        sqldict['Email'].append(mail)
                        
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')           
            
            os.remove(filePath)
            
        # Money Sevices Businesses    
        elif i == 7:
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
        
            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       
            
            for k in range(tables.n):
                df_Table = tables[k].df
                
                for j in range(len(df_Table)):
                    if 'NAME OF INSTITUTION' not in df_Table[0][j].upper() and df_Table[0][j].strip() != '' :
                        name = ' '.join([item.strip() for item in df_Table[0][j].split() if item !=''])
                        while name.split()[0] == 'A':
                            name = name[2:]
                        sqldict['Name'].append(name.replace('uthorized','Authorized').replace('frican','African').replace('dvance','Advance'))
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['ListCode'].append('8')
                        sqldict['RegCode'].append('FSRCAG')
                        sqldict['RegCtry'].append('AG')
                        sqldict['RegulationType'].append('Supervised')
                        
                        web = ''
                        mail = ''
                        tel = ''
                        fax = ''
                        address = ''
                        
                        for el in df_Table[2][j].splitlines():
                            if 'Email:' in el and 'Website:' in el:
                                mail = el.split('Website:')[0].replace('Email:','').strip()
                                web = el.split('Website:')[1].strip()
                            elif findUrl(el) != '':
                                web = findUrl(el)
                            elif findEmail(el) != '':
                                mail = findEmail(el)
                            elif len(patternTel.findall(el)) != 0:
                                tel = el.strip().replace(patternTel.findall(el)[0],'').replace(';','').replace('No.','').strip()
                            elif len(patternFax.findall(el)) != 0:
                                fax = el.replace(patternFax.findall(el)[0],'').replace('No.','').strip()
                            else:
                                if len(el.strip())>1 and 'CONTACT PERSON:' not in el.upper() and (len(findTel(el))<=2 or 'P.O.' in el):
                                    address = address +' '+el.strip()
                        
                        sqldict['Address_1'].append(address)
                        sqldict['Phone'].append(tel)
                        sqldict['Fax'].append(fax)
                        sqldict['Email'].append(mail)
                        
                        # Fill the rest with empty string
                        for key in sqldict.keys():
                            if len(sqldict['Name']) > len(sqldict[key]):
                                sqldict[key].append('')
            #os.remove(filePath)
        else:
            
            print(os.listdir(tempfolder))
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)
            
            os.remove(filePath)



df=pd.DataFrame(sqldict)
writer = ExcelWriter(filename)
df.to_excel(writer, 'SQL Ready', index=False)

writer.save()
writer.close()

sleep(3)

driver.quit()
    
    
    
    
    
    

<>:85: SyntaxWarning: invalid escape sequence '\-'
<>:102: SyntaxWarning: invalid escape sequence '\.'
<>:85: SyntaxWarning: invalid escape sequence '\-'
<>:102: SyntaxWarning: invalid escape sequence '\.'
C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_11368\3109477303.py:85: SyntaxWarning: invalid escape sequence '\-'
  regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')
C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_11368\3109477303.py:102: SyntaxWarning: invalid escape sequence '\.'
  patternTel = re.compile('(el.:|el\. |Tel:|Cell:|Tel.)')


Running AG FSRCAG Web Scraping Tool v.1.0
The directory tempfolder already exists.
Working with AG FSRCAG 1
Waiting for file to download
['Directory_of_Credit_Unions.pdf']


C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_11368\3109477303.py:85: SyntaxWarning: invalid escape sequence '\-'
  regex = re.compile('[a-zA-Z0-9.\-_]+@[^@]+\.[^@]+')
C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_11368\3109477303.py:102: SyntaxWarning: invalid escape sequence '\.'
  patternTel = re.compile('(el.:|el\. |Tel:|Cell:|Tel.)')


AttributeError: 'NoneType' object has no attribute 'splitlines'

In [2]:
df_Table

[['', 'Name of Institution', '', '', 'Address', '', '', 'Membership Type', ''],
 ['Antigua and Barbuda Co-operative League\nLtd.',
  'Antigua and Barbuda Co-operative League',
  None,
  '',
  'Cnr. Woods Centre and Mahogany Drive',
  '',
  'NA (Apex Body)',
  'NA (Apex Body)',
  None],
 [None, 'Ltd.', None, None, '3rd Floor', None, None, None, None],
 [None, '', None, None, 'St. John’s', None, None, None, None],
 [None, None, None, None, 'Antigua', None, None, None, None],
 [None, None, None, None, '', None, None, None, None],
 [None,
  None,
  None,
  None,
  'Tel.:(268) 462-9117 Fax.: (268) 562-2457',
  None,
  None,
  None,
  None],
 [None, None, None, None, '', None, None, None, None],
 [None, None, None, None, 'Dorothea Blackman Brown', None, None, None, None],
 [None, None, None, None, 'Chief Executive Officer', None, None, None, None],
 [None, None, None, None, '', None, None, None, None],
 [None, None, None, None, 'Email. dbbrown@abcl.ag', None, None, None, None],
 [None,
  Non

In [ ]:
def clean_list(lst):
    cleaned = []
    for i, item in enumerate(lst):
        # Skip if it's a prefix of the previous item
        if i > 0 and lst[i - 1].startswith(item):
            continue
        cleaned.append(item)
    return cleaned
for table in tables:
    for index,row in enumerate(table):
        # Check if any non-empty value exists in the row
        row_values = [str(cell) for cell in row if cell is not None]
        if any(row_values):
            if row_values[0] == '':
                continue
            else:
                row_values[0].replace('\n',' ')
            print(index)
            if len(row_values)>2:
                #print(row_values)
                clean_row_values = clean_list(row_values)
                print('Name:',clean_row_values[0].replace('\n',' '))
                print('Address:',clean_row_values[1])
                print('MembershipType:', clean_row_values[-1])
            else:
                if 'Email' in row_values[0]:
                    print('Email: ',row_values[0].split(' ')[-1])
                elif 'Tel' in row_values[0]:
                    print('Tel: ',row_values[0].split('Tel')[-1].strip().removeprefix('.:'))
                elif 'Website' in row_values[0]:
                    print('Website:',row_values[0].split('Website')[-1].strip().removeprefix(':'))
            
            

1
Name: Antigua and Barbuda Co-operative League Ltd.
Address: Cnr. Woods Centre and Mahogany Drive
MembershipType: NA (Apex Body)
2
4
6
Tel:  (268) 462-9117 Fax.: (268) 562-2457
8
9
11
Email:  dbbrown@abcl.ag
12
Website:  http://www.antiguacoopleague.com/
13
Name: APUA Co-operative Credit Union
Address: (The Old APUA Head Office Building)
MembershipType: Closed Bond
14
15
16
18
Tel:   (268) 480-7126 / (268) 480-7797
20
21
23
Email:  apuaccu@apua.ag
1
Name: Christian Co-operative Credit Union
Address: Bishopsgate Street
MembershipType: Open Bond
2
3
5
Tel:   (268) 562-6114
7
8
10
Email:  christiancooperative2004@gmail.com
11
Name: Seventh Day Adventist Co-operative Credit Union
Address: All Saints Road
MembershipType: Closed Bond
12
13
15
Tel:  . (268) 562 – 3446
17
18
20
Email:  nwalker@sdaccul.com
21
Name: Community First Co-operative Credit Union
Address: Old Parham Road
MembershipType: Open Bond
22
23
25
Tel:   (268) 481-3950
27
28
29
30
32
Tel:  : (268)481-4000
34
35
2
Website:  ht

: 

In [40]:
# Create a function to remove duplicates while preserving order
def remove_duplicates(lst):
    seen = set()
    unique_list = []
    for item in lst:
        # Convert row values to tuple so it can be added to set
        # Skip empty items and check if not seen before
        if item and tuple(item) not in seen:
            seen.add(tuple(item))
            unique_list.append(item)
    return unique_list

# Apply the function to remove duplicates
clean_row_values = remove_duplicates(row_values)

In [49]:
def clean_list(lst):
    cleaned = []
    for i, item in enumerate(lst):
        # Skip if it's a prefix of the previous item
        if i > 0 and lst[i - 1].startswith(item):
            continue
        cleaned.append(item)
    return cleaned

In [8]:
for el in df_Table[1]:
    print(el)

Antigua and Barbuda Co-operative League
Ltd.
Antigua and Barbuda Co-operative League
None

Cnr. Woods Centre and Mahogany Drive

NA (Apex Body)
NA (Apex Body)
None
